# 05 - Sub-experiment 1: Zero-shot vs Trained Models

This notebook runs basic zero-shot classification on the shared 200-post sample and compares results against LR, SVM, and MLP from the saved CV summary.

In [1]:
import os
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
import sys
sys.modules["tensorflow"] = None

### What this does and why

This is the baseline LLM comparison. We keep the labels simple and test whether a locally-run BART zero-shot model can compete with the previously trained classical models.

In [2]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from tqdm import tqdm
from transformers import pipeline
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

root = Path.cwd()
outputs_dir = root / "outputs"
if not outputs_dir.exists():
    outputs_dir = root.parent / "outputs"

sample_df = pd.read_csv(outputs_dir / "llm_sample.csv")
cv = pd.read_csv(outputs_dir / "reddit_cv_summary.csv")

def compute_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    p_w, r_w, f1_w, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)
    p_m, r_m, f1_m, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    return {
        "accuracy": float(acc),
        "precision_weighted": float(p_w),
        "recall_weighted": float(r_w),
        "f1_weighted": float(f1_w),
        "precision_macro": float(p_m),
        "recall_macro": float(r_m),
        "f1_macro": float(f1_m),
    }

def create_zero_shot_pipeline():
    candidate_models = [
        "typeform/distilbert-base-uncased-mnli",
        "valhalla/distilbart-mnli-12-1",
        "facebook/bart-large-mnli",
    ]
    for model_name in candidate_models:
        try:
            return pipeline(
                "zero-shot-classification",
                model=model_name,
                device=-1,
            )
        except Exception as e:
            print(f"Could not load zero-shot model {model_name}: {e}")
    raise RuntimeError(
        "Failed to load any zero-shot classification model. "
        "Try increasing system paging/swap or use a smaller model."
    )

classifier = create_zero_shot_pipeline()

labels = ["mental health risk", "high risk suicidal"]
y_true = sample_df["risk_label"].astype(int).to_numpy()
texts = sample_df["text_clean"].astype(str).tolist()
preds, confs = [], []
for t in tqdm(texts, desc="Basic zero-shot inference"):
    out = classifier(t, candidate_labels=labels)
    winner = out["labels"][0]
    score = float(out["scores"][0])
    pred = 0 if winner == labels[0] else 1
    preds.append(pred)
    confs.append(score)

preds = np.array(preds, dtype=int)
m = compute_metrics(y_true, preds)
(outputs_dir / "zeroshot_basic_metrics.json").write_text(json.dumps(m, indent=2), encoding="utf-8")

trained_rows = cv.loc[cv["model"].isin(["LR", "SVM", "MLP"]), ["model", "accuracy_mean", "f1_weighted_mean", "f1_macro_mean"]].copy()
trained_rows["Type"] = "Trained"
trained_rows = trained_rows.rename(columns={
    "model": "Model",
    "accuracy_mean": "Accuracy",
    "f1_weighted_mean": "Weighted F1",
    "f1_macro_mean": "Macro F1",
})

zs_row = pd.DataFrame([{
    "Model": "BART-ZS",
    "Type": "Zero-Shot",
    "Accuracy": m["accuracy"],
    "Weighted F1": m["f1_weighted"],
    "Macro F1": m["f1_macro"],
}])
comparison = pd.concat([trained_rows[["Model", "Type", "Accuracy", "Weighted F1", "Macro F1"]], zs_row], ignore_index=True)
comparison.to_csv(outputs_dir / "zeroshot_vs_trained_comparison.csv", index=False)

cm = confusion_matrix(y_true, preds, labels=[0, 1])
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Mental Health Risk", "High Risk Suicidal"], yticklabels=["Mental Health Risk", "High Risk Suicidal"])
plt.title("Confusion Matrix - Zero-shot (Basic)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.savefig(outputs_dir / "cm_zeroshot_basic.png", dpi=200)
plt.close()

plt.figure(figsize=(7, 4))
sns.barplot(data=comparison, x="Model", y="Weighted F1", hue="Type")
plt.ylim(0, 1)
plt.title("Weighted F1 - Trained vs Zero-shot")
plt.tight_layout()
plt.savefig(outputs_dir / "zeroshot_vs_trained_f1.png", dpi=200)
plt.close()

pred_df = sample_df[["row_id", "text_clean", "risk_label"]].copy()
pred_df["zs_basic_pred"] = preds
pred_df["zs_basic_confidence"] = confs
pred_df.to_csv(outputs_dir / "zeroshot_basic_predictions.csv", index=False)

comparison

C:\Users\HP\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Basic zero-shot inference: 100%|██████████| 200/200 [00:45<00:00,  4.43it/s]


,Model,Type,Accuracy,Weighted F1,Macro F1
0,SVM,Trained,0.78875,0.788726,0.788726
1,MLP,Trained,0.76400,0.763938,0.763938
2,LR,Trained,0.75725,0.757150,0.757150
3,BART-ZS,Zero-Shot,0.56500,0.545917,0.545917
